# VITS — Run Locally with Jupyter


Source and license: [VITS](https://github.com/jaywalnut310/vits), MIT. See `src/tts/vits/NOTICE.md` for details.

In [ ]:
from pathlib import Path
import os
import sys
import json
import signal
import subprocess
import torch
from IPython.display import Audio, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/vits.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the English-TTS project")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Project:", ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "Device:", DEVICE)
print("MPS available:", torch.backends.mps.is_available(), "(not enabled in this integration)")
if DEVICE == "cpu":
    print("CPU: suitable for checks; full VITS training is very slow.")

## Run Commands

Training output appears in this notebook. During training, the Stop/Interrupt button requests a graceful stop. Wait for the current step to finish and for confirmation that `last.pt` has been saved. Forcing the kernel to shut down may lose progress since the last checkpoint.

In [ ]:
from collections import deque

PROJECT_PYTHON = ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not PROJECT_PYTHON.is_file():
    raise RuntimeError(f"Project Python not found: {PROJECT_PYTHON}")
print("Script interpreter:", PROJECT_PYTHON)

def run_module(module, *arguments):
    command = [str(PROJECT_PYTHON), "-u", "-m", module, *map(str, arguments)]
    environment = os.environ.copy()
    # An activated Conda/Jupyter kernel must not override the project's venv.
    environment.pop("PYTHONHOME", None)
    environment.pop("PYTHONPATH", None)
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True,
                               env=environment)
    recent_output = deque(maxlen=60)
    def show_output():
        for line in process.stdout:
            recent_output.append(line)
            print(line, end="", flush=True)
    try:
        show_output()
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
        print("Waiting for the current step and checkpoint save...")
        show_output()
    finally:
        code = process.wait()
        process.stdout.close()
    if code:
        details = "".join(recent_output)
        raise RuntimeError(f"{module} exited with code {code}\nPython: {PROJECT_PYTHON}\n{details}")

## 1. Prepare LJSpeech for VITS

Use the existing WAV files and the official phoneme transcriptions. The previous model's mel cache is not compatible with VITS. This cell can be run again with the same settings.

In [ ]:
DATA_ROOT = ROOT / "data/raw/LJSpeech-1.1"
assert (DATA_ROOT / "metadata.csv").is_file(), "Extract LJSpeech into data/raw/LJSpeech-1.1 first"
run_module("scripts.prepare_vits", "--data-root", DATA_ROOT)
metadata = json.loads((ROOT / "data/processed/vits/metadata.json").read_text())
print("Architecture:", metadata["architecture"], "Vocabulary:", len(metadata["symbols"]))

## 2. Smoke Test with a Small VITS Model

Run two training steps, save a checkpoint, then resume for a third step. This small model is only for testing and is not used for full training. Each run creates a separate directory. Its checkpoint is not compatible with the full model.

In [ ]:
RUN_SMOKE = True
SMOKE_CHECKPOINT = None
if RUN_SMOKE:
    from datetime import datetime
    smoke_output = ROOT / "checkpoints" / ("vits-smoke-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
    run_module("scripts.prepare_vits", "--data-root", DATA_ROOT,
               "--config", "configs/vits-smoke.json", "--output", "data/processed/vits-smoke")
    common = ["--config", "configs/vits-smoke.json", "--data", "data/processed/vits-smoke",
              "--output", smoke_output, "--device", DEVICE, "--threads", "2"]
    run_module("scripts.train_vits", *common, "--max-steps", "2")
    SMOKE_CHECKPOINT = smoke_output / "last.pt"
    run_module("scripts.train_vits", *common, "--resume", SMOKE_CHECKPOINT, "--max-steps", "3")
    run_module("scripts.infer_vits", "--checkpoint", SMOKE_CHECKPOINT,
               "--text", "Hello world.", "--output", "outputs/vits-smoke.wav", "--max-frames", "100", "--device", DEVICE)
    print("Smoke test completed. This waveform is untrained noise, not a quality result.")

## 3. Train the Full Model

Set `RUN_TRAINING = True` to start. `EPOCHS` is the total target number of epochs; a value of 1 is only an initial check and is not enough to produce high-quality speech.

If `last.pt` exists, training resumes automatically with the same settings. Checkpoints are saved every 100 steps, at the end of each epoch, and after a graceful stop. `best.pt` is selected using reconstruction mel loss on up to 8 validation batches; this is not a measure of perceptual speech quality. Keep the batch size unchanged when resuming.

In [ ]:
RUN_TRAINING = True
EPOCHS = 2
BATCH_SIZE = 2
OUTPUT = ROOT / "checkpoints/vits"
if RUN_TRAINING:
    arguments = ["--config", "configs/vits.json", "--data", "data/processed/vits",
                 "--output", OUTPUT, "--device", DEVICE,
                 "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE)]
    if (OUTPUT / "last.pt").is_file():
        arguments += ["--resume", OUTPUT / "last.pt"]
    run_module("scripts.train_vits", *arguments)
else:
    print("Training is off. Set RUN_TRAINING=True when ready.")

## 4. View Recent Losses

VITS reports generator, discriminator, mel, KL, duration, and feature-matching losses. These values are not directly comparable to the previous model's loss.

In [ ]:
metrics = OUTPUT / "metrics.jsonl"
if metrics.is_file():
    from collections import deque
    with metrics.open() as handle:
        recent = deque(handle, maxlen=10)
    for row in recent:
        entry = json.loads(row)
        print("step", entry["step"], " ".join(f"{k}={entry[k]:.4f}" for k in ["g", "d", "mel", "kl", "duration"]))
else:
    print("No full-model training metrics yet.")

## 5. Generate and Play Speech

After training, set `RUN_SYNTHESIS = True` and edit the English input text. The VITS neural vocoder generates the waveform directly.

In [ ]:
RUN_SYNTHESIS = True
TEXT = "hello"
CHECKPOINT = OUTPUT / "best.pt" if (OUTPUT / "best.pt").exists() else OUTPUT / "last.pt"
if RUN_SYNTHESIS:
    assert CHECKPOINT.is_file(), "Train the full model first"
    run_module("scripts.infer_vits", "--checkpoint", CHECKPOINT,
               "--text", TEXT, "--output", "outputs/vits.wav", "--device", DEVICE)
    display(Audio(filename=str(ROOT / "outputs/vits.wav")))
else:
    print("Synthesis is off. Enable after training.")